# 05 - RAG Evaluation with MLflow

This notebook evaluates the RAG chain using MLflow's GenAI evaluation framework.

**Evaluation Dimensions:**
1. **Relevance** - Are retrieved chunks relevant to the query?
2. **Faithfulness/Groundedness** - Is the answer grounded in the retrieved context?
3. **Answer Correctness** - Does the answer correctly address the question?

**Approach:**
- Create an evaluation dataset with sample questions
- Run the RAG chain on each question (generating traces)
- Score using MLflow's built-in RAG judges
- Review results in the MLflow Experiment UI

In [0]:
%pip install -U mlflow databricks-langchain databricks-vectorsearch langchain langchain-core
dbutils.library.restartPython()

In [0]:
%run ./00_config

In [0]:
# =============================================================================
# MLFLOW SETUP
# =============================================================================

import mlflow

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
mlflow.langchain.autolog()

print(f"✅ MLflow experiment: {MLFLOW_EXPERIMENT_NAME}")

In [0]:
# =============================================================================
# REBUILD RAG CHAIN (for evaluation context)
# =============================================================================
# We rebuild the chain here to have access to it locally for evaluation.
# In production, you'd evaluate against the deployed endpoint instead.
# =============================================================================

from databricks_langchain import ChatDatabricks, DatabricksVectorSearch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()
vs_index = vsc.get_index(
    endpoint_name=VECTOR_SEARCH_ENDPOINT,
    index_name=VECTOR_INDEX_NAME
)

retriever = DatabricksVectorSearch(
    index=vs_index,
    text_column="chunk_to_retrieve",
    columns=["chunk_to_retrieve", "source_file", "chunk_id"]
).as_retriever(search_kwargs={"k": NUM_RESULTS})

llm = ChatDatabricks(
    endpoint=LLM_MODEL,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS
)

SYSTEM_PROMPT = """You are a knowledgeable assistant for project documentation. 
Answer questions based ONLY on the provided context. If the context does not contain 
relevant information to answer the question, say "I don't have enough information 
in the documentation to answer this question."

Always cite your sources by mentioning the document name.
Be concise, accurate, and helpful.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])

def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source_file", "Unknown")
        content = doc.page_content
        formatted.append(f"[Source {i}: {source}]\n{content}")
    return "\n\n---\n\n".join(formatted)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain rebuilt for evaluation.")

In [0]:
# =============================================================================
# EVALUATION DATASET
# =============================================================================
# Define a set of representative questions that cover different documents.
# For production, expand this with domain expert-curated Q&A pairs.
# =============================================================================

import pandas as pd

eval_dataset = pd.DataFrame({
    "inputs": [
        "What is the Broadcast pipeline and what are its main components?",
        "Describe the Fetch Likelihood preprocessing steps.",
        "What does the ComScore pipeline do?",
        "Explain the Hispanic Household Prediction model.",
        "What is the functional architecture of the Fetch Likelihood flow?",
        "How does the Broadcast Run and Refresh process work?",
        "What is the Broadcast Sustain process?",
        "Describe the end-to-end Broadcast pipeline.",
        "What is the Hispanic Modeling approach used in CDNA?",
        "What does the ECOMM Hispanic Modeling document cover?"
    ]
})

print(f"📊 Evaluation dataset: {len(eval_dataset)} questions")
eval_dataset.head()

In [0]:
# =============================================================================
# PREDICTION FUNCTION FOR EVALUATION
# =============================================================================
# Wraps the RAG chain in a function compatible with mlflow.genai.evaluate()
# =============================================================================

@mlflow.trace(span_type="CHAIN")
def predict_fn(inputs: dict) -> str:
    """Run the RAG chain and return the response."""
    question = inputs.get("inputs", inputs) if isinstance(inputs, dict) else inputs
    return rag_chain.invoke(question)

In [0]:
# =============================================================================
# RUN EVALUATION WITH MLFLOW GENAI SCORERS
# =============================================================================
# Uses MLflow's built-in RAG judges to assess:
# - Relevance: Are retrieved docs relevant to the query?
# - Groundedness: Is the answer faithful to the retrieved context?
# - Safety: Does the answer avoid harmful content?
# =============================================================================

from mlflow.genai.scorers import (
    Relevance,
    Groundedness,
    Safety
)

# Run evaluation
eval_results = mlflow.genai.evaluate(
    data=eval_dataset,
    predict_fn=predict_fn,
    scorers=[
        Relevance(),      # Is the response relevant to the question?
        Groundedness(),   # Is the response grounded in retrieved context?
        Safety()          # Is the response safe and appropriate?
    ]
)

print("✅ Evaluation complete!")
print(f"\n📊 Results Summary:")
print(eval_results.metrics)

In [0]:
# =============================================================================
# DISPLAY DETAILED RESULTS
# =============================================================================
# View per-question scores and rationales from the judges.
# Navigate to the MLflow Experiment UI for interactive exploration.
# =============================================================================

results_df = eval_results.tables["eval_results"]

# Display results
print("\n📝 Detailed Evaluation Results:")
print("=" * 80)
display(results_df)

print(f"\n\n🔗 View full results in MLflow Experiment UI:")
print(f"   Experiment: {MLFLOW_EXPERIMENT_NAME}")